# FUT.AI · Fase 4 — Homografía dinámica y métricas exploratorias

**Objetivo:** transformar las cajas de YOLO + ByteTrack a metros en distintos fotogramas, compensando cambios de cámara mediante seguimiento de referencias visuales del campo (Optical Flow + RANSAC).

**No vuelve a ejecutar YOLO ni solicita clics.** Parte del video MP4, el CSV de tracking y dos JSON de calibración manual (5 y 15 segundos).

**Importante:** esta es una prueba técnica, no una certificación de velocidades físicas. El cálculo se bloquea cuando el seguimiento de referencias se pierde. Un error de flujo bajo tampoco verifica por sí solo la exactitud absoluta de la homografía. Los CSV de métricas son estimaciones provisionales hasta validación independiente.

Ejecuta las celdas en orden. Cada celda imprime un resultado; ninguna queda esperando clics.

## Paso 1 — Preparar los cuatro archivos

En el panel **Archivos** de Colab (ícono de carpeta) sube el MP4, el CSV y los dos JSON de `prueba2_20s`. Si ya están cargados, no hace falta subirlos de nuevo. Se buscan los JSON por su contenido (`frame_calibrado`), aunque el nombre lleve `(1)`.

> Si Colab reinició la sesión, también tendrás que subir el video original de nuevo. Los minimapas PNG no reemplazan al MP4.

In [ ]:
from pathlib import Path
import os, json, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

VIDEO = Path('/content/prueba2_20s.mp4')
CSV_TRACKS = Path('/content/prueba2_20s_yolo11s.csv')
DIR_JSON = Path('/content')  # Carpeta donde están los JSON descargados de Colab
SALIDA = Path('/content/futai/fase4_dinamica')
SALIDA.mkdir(parents=True, exist_ok=True)

ANCHO_ANALISIS = 960          # Optimizamos el Optical Flow sin alterar las coordenadas originales del CSV
MAX_FRAMES = 500             # Exclusivamente los clips de 20 s del MVP
INTERVALO_MUESTRA = 5       # 5 frames ≈ 0,2 s a 25 fps
UMBRAL_VELOCIDAD = 40.0    # km/h; regla heurística de filtrado, NO valor de verdad

for ruta in (VIDEO, CSV_TRACKS):
    if not ruta.exists():
        raise FileNotFoundError(f'Falta {ruta.name}. Súbelo en Archivos de Colab o modifica su ruta.')

jsons = []
for base in (DIR_JSON, Path('/content/futai'), Path('/content/futai/diagnostico')):
    if base.exists():
        for p in base.glob('*.json'):
            try:
                d = json.loads(p.read_text())
                if (d.get('video', '').split('/')[-1] == VIDEO.name
                        and 'H_pixel_a_metros' in d and 'frame_calibrado' in d):
                    jsons.append((p, d))
            except Exception:
                pass

calibs = {}
for ruta, obj in jsons:
    frame = int(obj['frame_calibrado'])
    # Si hay duplicados, priorizar el archivo con más referencias sin mezclar el mismo frame.
    if frame not in calibs or len(obj.get('referencias', [])) > len(calibs[frame]['referencias']):
        calibs[frame] = obj

if not {125, 375}.issubset(calibs):
    raise FileNotFoundError('Se necesitan dos JSON de prueba2_20s: frame 125 (5 s) y frame 375 (15 s).')

tracks = pd.read_csv(CSV_TRACKS)
requeridas = {'frame','tiempo_s','track_id','x1','y1','x2','y2'}
if not requeridas.issubset(tracks.columns):
    raise ValueError('CSV sin columnas requeridas: ' + str(requeridas - set(tracks.columns)))

print('MP4:', VIDEO)
print('CSV:', len(tracks), 'observaciones')
for i in sorted(calibs):
    print(f'Calibración frame {i}: {len(calibs[i]["referencias"])} referencias; RMSE {calibs[i]["rmse_ajuste_m"]:.3f} m')

## Paso 2 — Cargar fotogramas a tamaño reducido y aislar el césped

Para ahorrar memoria, este prototipo guarda únicamente imágenes en escala de grises y máscaras binarias. La máscara intenta identificar el **campo verde** e ignorar jugadores según las cajas del CSV. Evita que el Optical Flow siga principalmente público o publicidad.

**Limitación:** si el césped tiene poco detalle o el video cambia bruscamente de plano, puede no haber suficientes puntos estáticos. En ese caso el algoritmo se detendrá en ese tramo.

In [ ]:
from collections import defaultdict

cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise RuntimeError('No se pudo abrir el video')
fps = cap.get(cv2.CAP_PROP_FPS)
w0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h0 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if fps <= 0 or w0 <= 0 or h0 <= 0:
    cap.release()
    raise ValueError('Metadatos inválidos del MP4')

scale = min(1.0, ANCHO_ANALISIS / w0)
w = round(w0 * scale)
h = round(h0 * scale)
sx, sy = w/w0, h/h0
S = np.diag([sx, sy, 1.0]).astype(np.float64)
SI = np.linalg.inv(S)

bbox_frame = defaultdict(list)
for r in tracks.itertuples(index=False):
    if 0 <= int(r.frame) < MAX_FRAMES:
        bbox_frame[int(r.frame)].append((float(r.x1),float(r.y1),float(r.x2),float(r.y2)))

# Características estáticas: buscar dentro de la mayor superficie verde de la escena.
def mascara_campo(bgr, cajas):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    verde = cv2.inRange(hsv, (28, 25, 25), (95, 255, 255))
    verde = cv2.morphologyEx(verde, cv2.MORPH_CLOSE, np.ones((15,15), np.uint8))
    verde = cv2.morphologyEx(verde, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    contours, _ = cv2.findContours(verde, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(verde)
    if contours:
        cont = max(contours, key=cv2.contourArea)
        if cv2.contourArea(cont) > 0.10 * (w*h):
            # Convex hull incluye líneas blancas dentro del terreno
            cv2.fillConvexPoly(mask, cv2.convexHull(cont), 255)
    mask = cv2.erode(mask, np.ones((11,11),np.uint8))
    for x1,y1,x2,y2 in cajas:
        a,b,c,d = round(x1*sx),round(y1*sy),round(x2*sx),round(y2*sy)
        pad = 6
        cv2.rectangle(mask, (max(0,a-pad),max(0,b-pad)), (min(w-1,c+pad),min(h-1,d+pad)), 0,-1)
    return mask

gris, mascaras = [], []
for k in range(MAX_FRAMES):
    ok, img = cap.read()
    if not ok:
        break
    small = cv2.resize(img, (w,h), interpolation=cv2.INTER_AREA)
    gris.append(cv2.cvtColor(small,cv2.COLOR_BGR2GRAY))
    mascaras.append(mascara_campo(small, bbox_frame[k]))
cap.release()
N = len(gris)
if N <= 375:
    raise ValueError('El MP4 no incluye el fotograma 375, por tanto no corresponde a estos JSON')

print(f'Video: {N} frames, {fps:.2f} FPS, tamaño {w0}x{h0} -> {w}x{h}')
plt.figure(figsize=(13,4))
plt.subplot(121);plt.imshow(gris[125],cmap='gray');plt.title('Frame 125');plt.axis('off')
plt.subplot(122);plt.imshow(mascaras[125],cmap='gray');plt.title('Máscara del campo (blanco = utilizable)');plt.axis('off')
plt.tight_layout();plt.show()

## Paso 3 — Calcular homografías por fotograma

Seguimos puntos del campo desde las calibraciones 125 y 375 hacia los frames cercanos (antes y después). Cada nueva imagen se compara con los puntos originarios del fotograma de referencia utilizando RANSAC.

**Regla de seguridad:** cuando se pierden demasiados puntos, no se asigna homografía al fotograma; no rellenamos el vacío con interpolaciones inventadas. Puedes añadir calibraciones manuales intermedias si quedan demasiados huecos.

In [ ]:
from dataclasses import dataclass

MAX_ERROR_FLUJO = 1.7          # píxeles a resolución reducida
MAX_ERROR_RANSAC = 3.0        # píxeles a resolución reducida
MIN_INLIERS = 12
MIN_RATIO = 0.55
MIN_SPAN_X = 0.10            # extensión de referencias en la imagen
MIN_SPAN_Y = 0.07

@dataclass
class Calidad:
    anchor: int
    inliers: int
    ratio: float
    reproy_px: float
    estado: str

# H_frame[índice] mapea píxeles ORIGINALES de ese frame a metros.
H_frame = {}
calidad = {}

for anchor, d in calibs.items():
    H = np.asarray(d['H_pixel_a_metros'],dtype=np.float64)
    if H.shape != (3,3) or not np.isfinite(H).all():
        raise ValueError(f'H incorrecta en anchor {anchor}')
    H_frame[anchor] = H / H[2,2]
    calidad[anchor] = Calidad(anchor, len(d['referencias']), 1.0, 0.0, 'manual')

# Las referencias de 5 y 15 s se usan en sus mitades respectivas.
frontera = (125 + 375)//2
segmentos = [
    (125, 0, -1), (125, frontera, 1),
    (375, frontera, -1), (375, N-1, 1),
]
H_extra = {}  # guardar doble estimación justo en la frontera (comprobación cruzada)

def evaluar_matriz(H, pts):
    if H is None or not np.isfinite(H).all():
        return False
    if abs(np.linalg.det(H)) < 1e-9:
        return False
    # Rechazar homografías que deformen excesivamente el área local de referencias.
    center = np.median(pts, axis=0).reshape(1,1,2).astype(np.float32)
    delta = np.array([[[0,0]], [[20,0]], [[0,20]], [[20,20]]], np.float32) + center
    mapped = cv2.perspectiveTransform(delta, H.astype(np.float64)).reshape(-1,2)
    if not np.isfinite(mapped).all():
        return False
    side1 = np.linalg.norm(mapped[1]-mapped[0])/20.0
    side2 = np.linalg.norm(mapped[2]-mapped[0])/20.0
    return 0.40 < side1 < 2.5 and 0.40 < side2 < 2.5

def propagar(anchor, destino, paso):
    puntos = cv2.goodFeaturesToTrack(
        gris[anchor], maxCorners=850, qualityLevel=0.004,
        minDistance=6, blockSize=5, mask=mascaras[anchor]
    )
    if puntos is None or len(puntos) < MIN_INLIERS + 4:
        print(f'Anchor {anchor}: no hay suficientes características del césped. Se necesita otro frame de calibración.')
        return
    p_ancla = puntos.reshape(-1,2).astype(np.float32)
    p_prev = p_ancla.copy()
    prev = gris[anchor]
    for idx in range(anchor+paso, destino+paso, paso):
        if not (0 <= idx < N): break
        nxt, st, _ = cv2.calcOpticalFlowPyrLK(
            prev, gris[idx], p_prev.reshape(-1,1,2), None,
            winSize=(25,25), maxLevel=3,
            criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT, 35, 0.01)
        )
        if nxt is None:
            break
        bk, st_back, _ = cv2.calcOpticalFlowPyrLK(
            gris[idx], prev, nxt, None,
            winSize=(25,25), maxLevel=3,
            criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,35,0.01)
        )
        if bk is None:
            break
        p_cur = nxt.reshape(-1,2)
        p_bk = bk.reshape(-1,2)
        valido = (st.reshape(-1)>0) & (st_back.reshape(-1)>0)
        valido &= np.isfinite(p_cur).all(axis=1) & np.isfinite(p_bk).all(axis=1)
        valido &= (np.linalg.norm(p_prev-p_bk,axis=1) < MAX_ERROR_FLUJO)
        # Excluir referencias que caen fuera del terreno o sobre jugadores.
        px=np.clip(np.rint(np.nan_to_num(p_cur[:,0],nan=-1000)).astype(int),0,w-1)
        py=np.clip(np.rint(np.nan_to_num(p_cur[:,1],nan=-1000)).astype(int),0,h-1)
        valido &= (p_cur[:,0]>=0)&(p_cur[:,0]<w)&(p_cur[:,1]>=0)&(p_cur[:,1]<h)
        valido &= (mascaras[idx][py,px] > 0)
        p_cur, p_ancla = p_cur[valido], p_ancla[valido]
        if len(p_cur) < MIN_INLIERS + 2:
            break
        # La homografía puede no estar bien definida si todos los puntos se amontonan.
        if (np.ptp(p_cur[:,0]) < MIN_SPAN_X*w or
            np.ptp(p_cur[:,1]) < MIN_SPAN_Y*h):
            break
        T, inliers = cv2.findHomography(
            p_cur, p_ancla, cv2.RANSAC,
            MAX_ERROR_RANSAC, maxIters=2000, confidence=0.995
        )
        if T is None or inliers is None:
            break
        good = inliers.reshape(-1).astype(bool)
        nin = int(good.sum())
        ratio = float(good.mean())
        if nin < MIN_INLIERS or ratio < MIN_RATIO or not evaluar_matriz(T, p_cur[good]):
            break
        reproy = cv2.perspectiveTransform(p_cur[good].reshape(-1,1,2),T).reshape(-1,2)
        error = float(np.median(np.linalg.norm(reproy-p_ancla[good],axis=1)))
        if not np.isfinite(error) or error > 2.0:
            break
        Hcur = H_frame[anchor] @ SI @ T @ S
        if abs(Hcur[2,2]) < 1e-10 or not np.isfinite(Hcur).all():
            break
        Hcur /= Hcur[2,2]
        if idx in H_frame and idx != anchor:
            H_extra[(anchor,idx)] = Hcur.copy()
        else:
            H_frame[idx] = Hcur
            calidad[idx] = Calidad(anchor, nin, ratio, error, 'optical_flow')
        # Seguir únicamente las trayectorias consistentes con el campo estático.
        p_prev = p_cur[good].astype(np.float32)
        p_ancla = p_ancla[good]
        prev = gris[idx]

for a,fin,paso in segmentos:
    propagar(a,fin,paso)
    print(f'Ancla {a}, hasta frame {fin}, sentido {"adelante" if paso>0 else "atrás"}: avance registrado')

print(f'Fotogramas con homografía: {len(H_frame)}/{N} = {len(H_frame)/N:.1%}')
for a in sorted(calibs):
    subset=[i for i,c in calidad.items() if c.anchor==a]
    print('Ancla',a,':',len(subset),'frames válidos (incluido ancla)')

# Comparación entre ambas anclas en la frontera: desacuerdo de posiciones en metros.
if frontera in H_frame and (375,frontera) in H_extra:
    H_a=H_frame[frontera]
    H_b=H_extra[(375,frontera)]
    m=tracks[tracks.frame==frontera]
    if not m.empty:
        p=np.stack([(m.x1+m.x2).to_numpy()/2,m.y2.to_numpy()],axis=1).astype(np.float32)
        A=cv2.perspectiveTransform(p.reshape(-1,1,2),H_a).reshape(-1,2)
        B=cv2.perspectiveTransform(p.reshape(-1,1,2),H_b).reshape(-1,2)
        diferencia=float(np.median(np.linalg.norm(A-B,axis=1)))
        print(f'Desacuerdo de homografías en frame {frontera}: mediana {diferencia:.2f} m')
        if diferencia > 2.0:
            print('ADVERTENCIA: las dos calibraciones no concuerdan en la frontera. No conectaremos segmentos para métricas.')
else:
    print('No se pudo validar la frontera de las anclas. No conectaremos segmentos para métricas.')

## Paso 4 — Convertir todas las detecciones a coordenadas del campo

Cada fila conserva `track_id` y `frame`. Los fotogramas donde falló la homografía quedan como **NO VÁLIDOS**, con `x_m` / `y_m` vacíos. Así no ocultamos huecos ni errores.

In [ ]:
L = float(calibs[125]['largo_cancha_m'])
W = float(calibs[125]['ancho_cancha_m'])
if abs(L-float(calibs[375]['largo_cancha_m'])) > 0.001 or abs(W-float(calibs[375]['ancho_cancha_m'])) > 0.001:
    raise ValueError('Los JSON usan dimensiones de cancha diferentes')

pos = tracks.copy()
pos['x_m'] = np.nan
pos['y_m'] = np.nan
pos['calibracion_valida'] = False
pos['ancla'] = pd.Series(pd.NA,index=pos.index,dtype='Int64')
pos['en_cancha'] = False

for frame, grupo in pos.groupby('frame',sort=True):
    frame=int(frame)
    if frame not in H_frame:
        continue
    coords=np.column_stack(((grupo.x1.to_numpy()+grupo.x2.to_numpy())/2, grupo.y2.to_numpy())).astype(np.float64)
    coords_m=cv2.perspectiveTransform(coords.reshape(-1,1,2),H_frame[frame]).reshape(-1,2)
    idx=grupo.index
    pos.loc[idx,'x_m']=coords_m[:,0]
    pos.loc[idx,'y_m']=coords_m[:,1]
    pos.loc[idx,'calibracion_valida']=True
    pos.loc[idx,'ancla']=calidad[frame].anchor
    pos.loc[idx,'en_cancha']=(
        np.isfinite(coords_m).all(axis=1) &
        (coords_m[:,0]>=0) & (coords_m[:,0]<=L) &
        (coords_m[:,1]>=0) & (coords_m[:,1]<=W)
    )

csv_pos=SALIDA/'posiciones_dinamicas.csv'
pos.to_csv(csv_pos,index=False)
print('Archivo:',csv_pos)
print('Observaciones con calibración válida:',int(pos.calibracion_valida.sum()),'/',len(pos))
print('Observaciones en terreno y calibradas:',int(pos.en_cancha.sum()))
print('Fotogramas sin homografía:',N-len(H_frame))

plt.figure(figsize=(12,3))
plt.plot(range(N),[calidad[i].inliers if i in calidad else np.nan for i in range(N)],'.',markersize=3)
plt.axvline(125,color='gray',linestyle='--');plt.axvline(375,color='gray',linestyle='--')
plt.title('Puntos estáticos que sustentan cada homografía (huecos = sin calibración)')
plt.xlabel('Fotograma');plt.ylabel('Inliers');plt.grid(alpha=0.2);plt.show()

## Paso 5 — Animación del minimapa (salida MP4)

Los puntos conservan sus IDs. Los fotogramas sin homografía muestran un aviso en vez de posiciones inventadas. El campo se dibuja con 105 × 68 m según la hipótesis de calibración anterior.

In [ ]:
from IPython.display import display, Video

ESC=7; MARG=35
VW,VH = round(L*ESC)+2*MARG, round(W*ESC)+2*MARG
base=np.full((VH,VW,3),(50,107,46),np.uint8)


def pt(x,y): return (int(MARG+x*ESC),int(MARG+y*ESC))

def cancha_dibujo():
    img=base.copy()
    c=(235,235,235)
    cv2.rectangle(img,pt(0,0),pt(L,W),c,2)
    cv2.line(img,pt(L/2,0),pt(L/2,W),c,2)
    cv2.circle(img,pt(L/2,W/2),int(9.15*ESC),c,2)
    cv2.circle(img,pt(L/2,W/2),2,c,-1)
    for xleft,xright in ((0,16.5),(L-16.5,L)):
        cv2.rectangle(img,pt(xleft,(W-40.32)/2),pt(xright,(W+40.32)/2),c,2)
    return img

rows_por_frame={int(i):g for i,g in pos[pos.en_cancha].groupby('frame')}
video_mapa=SALIDA/'minimapa_dinamico.mp4'
vwriter=cv2.VideoWriter(str(video_mapa),cv2.VideoWriter_fourcc(*'mp4v'),fps,(VW,VH))
if not vwriter.isOpened():
    raise RuntimeError('No se pudo crear el MP4 del minimapa')
for idx in range(N):
    mapa=cancha_dibujo()
    cv2.putText(mapa,f'FUT.AI   {idx/fps:.2f} s', (12,21),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1)
    if idx not in H_frame:
        cv2.putText(mapa,'SIN HOMOGRAFIA VALIDA', (round(VW*.26),round(VH*.5)), cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,70,255),2)
    else:
        for r in rows_por_frame.get(idx,[]).itertuples(index=False) if idx in rows_por_frame else []:
            p=pt(r.x_m,r.y_m)
            cv2.circle(mapa,p,4,(50,80,255),-1)
            cv2.putText(mapa,str(int(r.track_id)),(p[0]+5,p[1]-5),cv2.FONT_HERSHEY_SIMPLEX,0.33,(255,255,255),1)
    vwriter.write(mapa)
vwriter.release()
print('Minimapa guardado:',video_mapa)
print('Puedes abrirlo desde el panel Archivos de Colab y descargarlo.')

## Paso 6 — Métricas **exploratorias**: distancia observada y velocidad

**No interpretes estos números como precisión deportiva validada.** Para reducir saltos de cajas, calculamos una mediana temporal de posiciones y medimos cada 5 frames. Los intervalos sin homografía, las salidas del campo, las pérdidas de track y el cambio de ancla **no se conectan**. Un ID de ByteTrack no necesariamente coincide con un jugador durante todo el video.

Solo se reportan **tramos observados válidos**. Distancia no significa distancia total del jugador en el partido. La velocidad máxima se calcula sobre muestras filtradas, por lo que es una estimación preliminar.

In [ ]:
registros=[]
for track_id, grupo in pos.sort_values('frame').groupby('track_id'):
    sub=grupo[grupo.en_cancha & grupo.calibracion_valida].copy().sort_values('frame')
    if sub.empty:
        continue
    # Reiniciar tramo ante huecos de detección o al cambiar de calibración base.
    es_corte=(sub.frame.diff().fillna(1)!=1) | (sub.ancla.ne(sub.ancla.shift(1)).fillna(True))
    sub['segmento']=es_corte.cumsum()
    for seg, g in sub.groupby('segmento'):
        if len(g)<max(12,INTERVALO_MUESTRA*2):
            continue
        g=g.copy().sort_values('frame')
        g['xs']=g.x_m.rolling(7,center=True,min_periods=3).median()
        g['ys']=g.y_m.rolling(7,center=True,min_periods=3).median()
        muestras=g.iloc[::INTERVALO_MUESTRA]
        if len(muestras)<3:
            continue
        xy=muestras[['xs','ys']].to_numpy(float)
        tt=muestras.tiempo_s.to_numpy(float)
        dt=np.diff(tt)
        dd=np.linalg.norm(np.diff(xy,axis=0),axis=1)
        vv=3.6*dd/np.maximum(dt,1e-6)
        good=np.isfinite(vv)&(dt>0)&(vv<=UMBRAL_VELOCIDAD)
        # Descartar segmentos con demasiados saltos inverosímiles
        if good.sum() == 0:
            continue
        registros.append({
            'track_id':int(track_id),
            'segmento':int(seg),
            'ancla':int(g.ancla.iloc[0]),
            'frame_inicial':int(g.frame.iloc[0]),
            'frame_final':int(g.frame.iloc[-1]),
            'duracion_observada_s':round(float(g.tiempo_s.iloc[-1]-g.tiempo_s.iloc[0]),2),
            'distancia_observada_filtrada_m':round(float(dd[good].sum()),2),
            'velocidad_max_estimada_kmh':round(float(vv[good].max()),2),
            'intervalos_descartados':int((~good).sum()),
            'estado':'PROVISIONAL_NO_VALIDADO',
        })

tramos = pd.DataFrame(registros)
archivo_tramos=SALIDA/'metricas_tramos_provisionales.csv'
tramos.to_csv(archivo_tramos,index=False)
if tramos.empty:
    print('No hay suficientes secuencias válidas para calcular métricas. Revisa los huecos de homografía y calibra un frame intermedio.')
else:
    resumen=(tramos.groupby('track_id',as_index=False)
        .agg(distancia_observada_m=('distancia_observada_filtrada_m','sum'),
             velocidad_maxima_estimada_kmh=('velocidad_max_estimada_kmh','max'),
             duracion_observada_s=('duracion_observada_s','sum'),
             numero_tramos=('segmento','count'))
        .sort_values('distancia_observada_m',ascending=False))
    resumen['estado']='PROVISIONAL_NO_VALIDADO'
    resumen.to_csv(SALIDA/'metricas_por_track_provisionales.csv',index=False)
    display(resumen.head(15))
    print('Resultados provisionales:',SALIDA/'metricas_por_track_provisionales.csv')
print('Detalle de tramos:',archivo_tramos)

## Paso 7 — Informe de calidad para la entrega

**Qué presentar:** capturas del frame de referencia, máscara del campo, gráfica de inliers, minimapa animado, CSV de posiciones y limitaciones. Una demo que muestra huecos honestamente es preferible a métricas falsas.

In [ ]:
reporte={
    'video':VIDEO.name,
    'fps':float(fps), 'frames':int(N),
    'frames_con_homografia':int(len(H_frame)),
    'porcentaje_frames_con_homografia':round(100*len(H_frame)/N,1),
    'calibraciones_manuales':sorted(map(int,calibs.keys())),
    'observaciones_tracking':int(len(pos)),
    'observaciones_en_cancha_y_calibradas':int(pos.en_cancha.sum()),
    'tiene_metricas_exploratorias':bool(not tramos.empty),
    'advertencias':[
        'Velocidades y distancias PROVISIONALES, no validadas contra mediciones reales.',
        'Un ID de tracker puede fragmentarse durante oclusiones.',
        'Optical Flow puede derivar con el movimiento de cámara; sus inliers no prueban exactitud absoluta.',
        'Las coordenadas de pies se aproximan al borde inferior de la caja.',
        'El campo 105x68 m es una hipótesis no verificada para este estadio.',
        'Los fotogramas sin homografía NO se interpolan; el recorrido observado puede subestimar la distancia real.',
    ],
}
reporte_path=SALIDA/'reporte_calidad.json'
reporte_path.write_text(json.dumps(reporte,indent=2,ensure_ascii=False))
print(json.dumps(reporte,indent=2,ensure_ascii=False))
print('Todos los archivos están en:',SALIDA)